# 04 — Media reconstruction (iMO1056 / PAO1)

## Scientific question
Can we encode **Lee MOPS-pyruvate**, **Turner MOPS-succinate**, and an honest
`LB_approx` as documented exchange configurations with provenance, and is
wild-type growth feasible under each?

## Scope
- Batch A DEFINED: `mops_pyruvate_lee2015`, `mops_succinate_turner2015`
- Batch B PARTIAL: `LB_approx_lee2015` (yeast extract / tryptone flagged)
- **No** M9 in this batch (no READY_ONLY PAO1 study Methods lock M9 here)
- **No** validation metrics in this notebook; MOPS-pyruvate/D-21 metrics exist in the separate V-01 report, pending interpretation

Heavy logic lives in `src/medium.py`. This notebook only loads, displays, and saves.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import cobra

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.medium import (
    iter_implemented_media,
    apply_medium,
    medium_to_frame,
    unmodeled_to_frame,
    export_media_tables,
    wildtype_growth,
)

MODEL = ROOT / 'models' / 'paeruginosa_MODEL1507180020.xml'
OUT = ROOT / 'data' / 'processed' / 'conditions'
print('ROOT', ROOT)
print('MODEL', MODEL.exists())


## Environment / model identity


In [ ]:
model = cobra.io.read_sbml_model(str(MODEL))
assert 'PA_Biomass6_DM' in str(model.objective.expression)
print(model.id, len(model.reactions), 'rxns;', len(model.genes), 'genes')
print('objective:', model.objective)


## Implemented media (provenance)
Each medium has citation + Methods/SI locator + exchange→bound sources.


In [ ]:
specs = list(iter_implemented_media())
summary = pd.DataFrame([
    {
        'medium_id': s.medium_id,
        'label': s.label,
        'strain_context': s.strain_context,
        'n_exchanges': len(s.exchanges),
        'n_unmodeled': len(s.unmodeled_components),
        'carbon': s.carbon_source_exchange,
        'locator': s.locator[:80] + '...',
    }
    for s in specs
])
summary


In [ ]:
for s in specs:
    print('=' * 72)
    print(s.medium_id, f'[{s.label}]')
    print('citation:', s.citation[:120], '...')
    print('unmodeled:', list(s.unmodeled_components))
    display(medium_to_frame(s).head(8))


## Apply media + record real WT growth (never invent)


In [ ]:
rows = []
for s in specs:
    status, growth = wildtype_growth(model, s, quiet=True)
    rows.append({'medium_id': s.medium_id, 'label': s.label,
                 'solver_status': status, 'wt_growth_per_h': growth})
    print(f'{s.medium_id}: {status}  growth={growth:.6f} /h')
growth_df = pd.DataFrame(rows)
growth_df


## Save derived outputs


In [ ]:
paths = export_media_tables(OUT)
growth_df.to_csv(OUT / 'media_wt_growth.csv', index=False)
for k, p in paths.items():
    print(k, '->', p)
print('growth ->', OUT / 'media_wt_growth.csv')


## Limitations / what we do **not** claim
- Concentration (mM) is documented from Methods/SI; uptake fluxes use the
  audited iMO1056 default-medium convention (−10 carbon, −1000 salts/O2),
  **not** a paper-derived concentration→flux formula.
- `LB_approx` is **PARTIAL**: yeast extract and tryptone are unmodeled; AA
  exchanges are a proxy only. Do not call it true LB.
- MOPS buffer / tricine / Ca²⁺ (no exchange) are flagged unmodeled.
- No precision/recall/F1/MCC here; see `project_management/audit/v01_mops_pyruvate/V01_BINARY_VALIDATION_REPORT.md` for the completed first binary evaluation.
- Teaching notebook `03` metrics remain forbidden.
